# Composite Detector Experiments

A dedicated experimentation notebook for **composite (ensembled) PII detectors**, separate from the main workshop notebook to keep the experiment surface area clean.

**What this notebook answers:**

1. **Individual baselines** — what F1 does each detector hit on its own?
2. **Local-only composite** — when we category-best ensemble every detector *except* Skyflow, how close can we get to Skyflow alone?
3. **All-in composite** — when we include Skyflow in the recipe, what's the ceiling?

For each composite we also report an **honest holdout score** (recipe fit on half the fixtures, scored on the other half) alongside the upper-bound score (recipe fit + scored on the full set). The gap between the two is your variance estimate.

Downstream goal: identify the recipe that makes sense to ship in the `opf-api` composite endpoint.

## 1. Setup

Same harness install as the main workshop notebook. Spacy first (Colab may prompt to restart), then the workspace packages.

In [ ]:
!python -m spacy download en_core_web_lg -q

# Uncomment for multilingual Presidio (~3 GB extra download):
# !python -m spacy download nl_core_news_lg -q
# !python -m spacy download fr_core_news_lg -q
# !python -m spacy download de_core_news_lg -q
# !python -m spacy download it_core_news_lg -q
# !python -m spacy download es_core_news_lg -q

print("\nspaCy models ready. If Colab prompts to restart the session, do it now and re-run from the top.")

In [ ]:
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_BRANCH = "main"   # set to a branch name to test a PR before merge
OPF_REPO = "https://github.com/openai/privacy-filter.git"

import os, subprocess, sys
os.environ.setdefault("OPF_MOE_TRITON", "0")

PIP = f"{sys.executable} -m pip"

def _run(cmd, *, msg, show_output=False):
    print(f"==> {msg}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 or show_output:
        if r.stdout: print(r.stdout)
        if r.stderr: print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{msg} failed (exit {r.returncode}). See output above.")

if not os.path.exists("/content/privacy-filter"):
    _run(f"git clone --depth 1 {OPF_REPO} /content/privacy-filter", msg="clone privacy-filter")
if not os.path.exists("/content/local-privacy"):
    _run(f"git clone --depth 1 --branch {HARNESS_BRANCH} {HARNESS_REPO} /content/local-privacy",
         msg=f"clone local-privacy@{HARNESS_BRANCH}")

_run(f"{PIP} install -q /content/privacy-filter", msg="install opf + deps")
_run(f"{PIP} install -q /content/local-privacy/eval", msg="install opf-eval + deps")
_run(f"{PIP} install -q /content/local-privacy/api", msg="install opf-api + deps")

for src_dir in ("/content/privacy-filter", "/content/local-privacy/eval/src", "/content/local-privacy/api/src"):
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

import importlib
for mod in ("opf", "opf_eval", "opf_eval.runner", "opf_eval.report", "opf_eval.ensemble", "opf_eval.fixtures", "opf_api"):
    importlib.import_module(mod)
    print(f"  ok: {mod}")

print("\nsetup complete.")

## 2. Configuration

**Default local detector set** matches what the main workshop notebook uses — fast on T4, well-tested. Override `LOCAL_DETECTORS` below if you want to throw more models at it.

`HOLDOUT_FRAC` controls the train/score split for the honest column. `0.5` is the conservative default; reducing it (e.g. `0.3`) leaves more for scoring at the cost of a noisier recipe.

In [ ]:
from pathlib import Path
import torch
from IPython.display import Markdown, display
from opf_eval import fixtures, runner, report, ensemble

# === EDIT THESE ===
DATASET = "pii_masking_200k"  # one of: pii_masking_300k, pii_masking_200k, pii_masking_400k, openpii_nano, openpii_mini
N_EXAMPLES = 200              # bump higher for stable signal; halving for holdout means each side gets N_EXAMPLES/2

# Local detectors are everything we can run on-box, no API calls.
# Default matches workshop notebook. Add to taste:
#   gliner_gretel_small / gliner_gretel_large — Gretel bi-encoder variants
#   ai4privacy_modernbert — ai4privacy ModernBERT, OpenPII vocab
#   openmed — DeBERTa per-language, snake_case vocab
LOCAL_DETECTORS = ["presidio", "gliner", "gliner_nvidia", "opf"]

# Skyflow is the cloud-only reference detector. Disable the next line if you
# don't have credentials — the rest of the notebook still runs, just without
# the Skyflow row + the all-in composite is identical to the local composite.
INCLUDE_SKYFLOW = True

ALL_DETECTORS = LOCAL_DETECTORS + (["skyflow"] if INCLUDE_SKYFLOW else [])

HOLDOUT_FRAC = 0.5
FIXTURE_SEED = 42
RUN_NAME = "composite_demo"
# ===================

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)

FIXTURES_PATH = Path(f"/content/data/{DATASET}_{N_EXAMPLES}.jsonl")
OUT_DIR = Path(f"/content/results/{RUN_NAME}")
FIXTURES_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"device:           {DEVICE}")
print(f"dataset:          {DATASET}")
print(f"n_examples:       {N_EXAMPLES} (holdout fits on {int(N_EXAMPLES*HOLDOUT_FRAC)}, scores on {N_EXAMPLES - int(N_EXAMPLES*HOLDOUT_FRAC)})")
print(f"local detectors:  {LOCAL_DETECTORS}")
print(f"all detectors:    {ALL_DETECTORS}")
print(f"fixtures path:    {FIXTURES_PATH}")
print(f"out dir:          {OUT_DIR}")

## 3. (Optional) Skyflow credentials

Load creds if `INCLUDE_SKYFLOW=True`. Auto-skips if not.

In [ ]:
if not INCLUDE_SKYFLOW:
    print("INCLUDE_SKYFLOW=False — skipping credential setup.")
else:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
        for var in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN"):
            try:
                os.environ[var] = userdata.get(var)
            except Exception:
                print(f"  {var}: not set in Colab Secrets")
        if all(v in os.environ for v in ("SKYFLOW_VAULT_URL", "SKYFLOW_VAULT_ID", "SKYFLOW_BEARER_TOKEN")):
            print("Skyflow creds loaded.")
        else:
            print("Skyflow creds incomplete. Either set them or flip INCLUDE_SKYFLOW=False in cell 2.")
    except ImportError:
        print("Not in Colab. Set SKYFLOW_* env vars in your shell.")

## 4. Materialize fixtures + split for holdout

Deterministic seed across the dataset → the same `N_EXAMPLES` every time. The holdout split is derived from `FIXTURE_SEED` so the recipe-fit and score halves are stable across runs.

In [ ]:
import json, random

if not FIXTURES_PATH.exists():
    n = fixtures.materialize(FIXTURES_PATH, N_EXAMPLES, dataset=DATASET, seed=FIXTURE_SEED)
    print(f"wrote {n} examples to {FIXTURES_PATH}")
else:
    print(f"reusing existing fixtures at {FIXTURES_PATH}")

FIXTURE_RECORDS = [json.loads(l) for l in FIXTURES_PATH.open() if l.strip()]
FIXTURE_IDS = [r["id"] for r in FIXTURE_RECORDS]

# Deterministic holdout split: shuffle ids under a derived seed, take the
# first HOLDOUT_FRAC for recipe-fit, the rest for scoring.
rng = random.Random(FIXTURE_SEED + 1)
shuffled = list(FIXTURE_IDS)
rng.shuffle(shuffled)
fit_n = int(len(shuffled) * HOLDOUT_FRAC)
FIT_IDS = set(shuffled[:fit_n])
SCORE_IDS = set(shuffled[fit_n:])

print(f"total ids:     {len(FIXTURE_IDS)}")
print(f"fit ids:       {len(FIT_IDS)}")
print(f"score ids:     {len(SCORE_IDS)}")
print(f"overlap (must be 0): {len(FIT_IDS & SCORE_IDS)}")

## 5. Run every detector on the full fixture set

One pass — every detector sees every fixture. The recipe-fit / scoring split happens later, in the composite-building step, by filtering ids in memory. No detector re-runs needed.

In [ ]:
runner.run(
    fixtures=FIXTURES_PATH,
    detector_names=ALL_DETECTORS,
    out_dir=OUT_DIR,
    dataset=DATASET,
    device=DEVICE,
    skyflow_workers=1,
    skyflow_min_interval_ms=0,
)

print("\nfiles written:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}")

## 6. Scoring helpers

Builds the comparison rows. `headline_f1(detector, score_ids, labels)` computes Strict / Exact / Partial / Type F1 against the gold spans of just those ids. Used by every composite section + the diff table at the end.

In [ ]:
from opf_eval.nervaluate_metrics import score as semeval_score
from opf_eval.taxonomy import dataset_canonicals
from opf_eval.datasets import get as get_dataset_config

_GOLD_BY_ID = {r["id"]: r["gold_spans"] for r in FIXTURE_RECORDS}
_VOCAB_KEY = get_dataset_config(DATASET).vocab_key
_DATASET_LABELS = sorted(dataset_canonicals(_VOCAB_KEY))
_LABEL_SET = set(_DATASET_LABELS)

SCHEMA_DISPLAY = [("strict", "Strict"), ("exact", "Exact"), ("partial", "Partial"), ("ent_type", "Type")]

def _load_raw(detector_name):
    path = OUT_DIR / f"raw_{detector_name}.jsonl"
    if not path.exists():
        return None
    return [json.loads(l) for l in path.open() if l.strip()]

def headline_f1(detector_name, score_ids, labels=None):
    """Compute SemEval F1 across all four schemas for `detector_name`,
    restricted to fixture ids in `score_ids` and labels in `labels`
    (defaults to the dataset's canonical set)."""
    labels = labels or _DATASET_LABELS
    label_set = set(labels)
    records = _load_raw(detector_name)
    if records is None:
        return None
    pairs = []
    for r in records:
        if r["id"] not in score_ids or r.get("error"):
            continue
        gold = [s for s in _GOLD_BY_ID.get(r["id"], []) if s["label"] in label_set]
        pred = [s for s in (r.get("spans") or []) if s["label"] in label_set]
        pairs.append((pred, gold))
    if not pairs:
        return None
    sem = semeval_score(detector=detector_name, pairs=pairs, tags=labels)
    return {schema: float(sem.by_schema.get(schema, {}).get("f1", 0.0)) for schema, _ in SCHEMA_DISPLAY}

def per_label_f1(detector_name, score_ids, labels=None):
    """Per-label Type-schema F1 — for the bar chart at the end."""
    labels = labels or _DATASET_LABELS
    label_set = set(labels)
    records = _load_raw(detector_name)
    if records is None:
        return {lbl: 0.0 for lbl in labels}
    pairs = []
    for r in records:
        if r["id"] not in score_ids or r.get("error"):
            continue
        gold = [s for s in _GOLD_BY_ID.get(r["id"], []) if s["label"] in label_set]
        pred = [s for s in (r.get("spans") or []) if s["label"] in label_set]
        pairs.append((pred, gold))
    if not pairs:
        return {lbl: 0.0 for lbl in labels}
    sem = semeval_score(detector=detector_name, pairs=pairs, tags=labels)
    return {lbl: float(sem.by_label.get(lbl, {}).get("ent_type", {}).get("f1", 0.0)) for lbl in labels}

print(f"helpers ready. dataset labels ({len(_DATASET_LABELS)}): {_DATASET_LABELS}")

## 7. Baseline — individual detectors

Headline F1 for every detector on its own, scored on the **full** fixture set. This is the apples-to-apples comparison; the composites below try to beat each row.

In [ ]:
ALL_IDS = set(FIXTURE_IDS)

baseline_rows = []
for det in ALL_DETECTORS:
    f1 = headline_f1(det, ALL_IDS)
    if f1 is None:
        continue
    baseline_rows.append((det, f1))

header = "| detector | " + " | ".join(label for _, label in SCHEMA_DISPLAY) + " |"
sep = "|" + "|".join(["---"] * (1 + len(SCHEMA_DISPLAY))) + "|"
lines = [header, sep]
for det, f1 in baseline_rows:
    cells = [f"`{det}`"] + [f"{f1[schema]:.3f}" for schema, _ in SCHEMA_DISPLAY]
    lines.append("| " + " | ".join(cells) + " |")

display(Markdown("### Headline F1 — individual detectors (scored on full fixture set)\n\n" + "\n".join(lines)))

## 8. Composite A — local detectors only (no Skyflow)

How close can the local-only composite get to Skyflow alone? The recipe picks the per-category-best detector from `LOCAL_DETECTORS`, then synthesises a new prediction stream.

Two variants:

- **Upper-bound** — recipe fit + scored on the full set. The number to beat.
- **Honest** — recipe fit on the holdout-fit half, scored on the holdout-score half. What you'd actually see on unseen data.

In [ ]:
# Upper-bound: recipe fit on full set.
local_full_path, local_full_recipe, local_full_f1s = ensemble.run_category_best(
    OUT_DIR,
    FIXTURES_PATH,
    dataset=DATASET,
    excluded_detectors={"skyflow", "skyflow_full"},
    ensemble_name="ensemble_local_full",
)

# Honest: recipe fit only on the FIT_IDS half.
local_holdout_path, local_holdout_recipe, local_holdout_f1s = ensemble.run_category_best(
    OUT_DIR,
    FIXTURES_PATH,
    dataset=DATASET,
    excluded_detectors={"skyflow", "skyflow_full"},
    ensemble_name="ensemble_local_holdout",
    fit_ids=FIT_IDS,
)

print("=== local-only recipe (upper-bound, fit on full set) ===")
for label in sorted(local_full_recipe):
    choices = sorted(local_full_f1s[label].items(), key=lambda kv: -kv[1])
    head = ", ".join(f"{d}={f:.2f}" for d, f in choices[:3])
    print(f"  {label:<14} -> {local_full_recipe[label]:<24}  (candidates: {head})")

print("\n=== local-only recipe (honest, fit on holdout-fit half) ===")
for label in sorted(local_holdout_recipe):
    choices = sorted(local_holdout_f1s[label].items(), key=lambda kv: -kv[1])
    head = ", ".join(f"{d}={f:.2f}" for d, f in choices[:3])
    print(f"  {label:<14} -> {local_holdout_recipe[label]:<24}  (candidates: {head})")

# Recipe drift between upper-bound and honest = your overfitting estimate.
drift = sorted(
    label for label in set(local_full_recipe) | set(local_holdout_recipe)
    if local_full_recipe.get(label) != local_holdout_recipe.get(label)
)
print(f"\nrecipe drift between full and holdout-fit: {len(drift)} labels — {drift}")

## 9. Composite B — all detectors (including Skyflow)

Same two variants — upper-bound + honest — but now Skyflow is in the candidate pool. If Skyflow dominates many categories the all-in composite will just be "Skyflow with a few local picks bolted on." If not, the local detectors can hold their own on specific categories.

In [ ]:
if not INCLUDE_SKYFLOW:
    print("INCLUDE_SKYFLOW=False — skipping all-in composite (identical to local-only).")
else:
    all_full_path, all_full_recipe, all_full_f1s = ensemble.run_category_best(
        OUT_DIR,
        FIXTURES_PATH,
        dataset=DATASET,
        ensemble_name="ensemble_all_full",
    )
    all_holdout_path, all_holdout_recipe, all_holdout_f1s = ensemble.run_category_best(
        OUT_DIR,
        FIXTURES_PATH,
        dataset=DATASET,
        ensemble_name="ensemble_all_holdout",
        fit_ids=FIT_IDS,
    )

    print("=== all-in recipe (upper-bound) ===")
    for label in sorted(all_full_recipe):
        choices = sorted(all_full_f1s[label].items(), key=lambda kv: -kv[1])
        head = ", ".join(f"{d}={f:.2f}" for d, f in choices[:3])
        print(f"  {label:<14} -> {all_full_recipe[label]:<24}  (candidates: {head})")

    print("\n=== all-in recipe (honest) ===")
    for label in sorted(all_holdout_recipe):
        choices = sorted(all_holdout_f1s[label].items(), key=lambda kv: -kv[1])
        head = ", ".join(f"{d}={f:.2f}" for d, f in choices[:3])
        print(f"  {label:<14} -> {all_holdout_recipe[label]:<24}  (candidates: {head})")

    skyflow_categories = sorted(l for l, d in all_full_recipe.items() if d == "skyflow")
    local_won_categories = sorted(l for l, d in all_full_recipe.items() if d != "skyflow")
    print(f"\ncategories where Skyflow won the recipe (upper-bound): {len(skyflow_categories)} — {skyflow_categories}")
    print(f"categories where a local detector beat Skyflow:         {len(local_won_categories)} — {local_won_categories}")

## 10. Headline comparison table

One table to rule them all. Compares:

- **Best individual local** — the highest-F1 single local detector (computed dynamically per schema, so a different detector may win each column)
- **Skyflow alone** (if INCLUDE_SKYFLOW)
- **Composite local-only** (upper-bound + honest)
- **Composite all-in** (upper-bound + honest)

`Δ vs Skyflow` columns show the gap to Skyflow alone (positive = composite beats Skyflow).

In [ ]:
# Compute every row's headline F1.
# Note on score-set choice: upper-bound rows score on the full set (same set
# the recipe was fit on — by definition an upper bound). Honest rows score
# only on SCORE_IDS so the recipe sees no signal from those records.

rows = []

# Best individual local — pick the highest local detector PER schema.
local_f1s = {det: headline_f1(det, ALL_IDS) for det in LOCAL_DETECTORS}
local_f1s = {d: f for d, f in local_f1s.items() if f is not None}
best_per_schema = {}
for schema, _ in SCHEMA_DISPLAY:
    best_det = max(local_f1s, key=lambda d: local_f1s[d][schema])
    best_per_schema[schema] = (best_det, local_f1s[best_det][schema])
rows.append((
    "best individual local (per-schema winner)",
    {schema: best_per_schema[schema][1] for schema, _ in SCHEMA_DISPLAY},
    {schema: best_per_schema[schema][0] for schema, _ in SCHEMA_DISPLAY},
))

# Skyflow baseline (if available).
skyflow_f1 = None
if INCLUDE_SKYFLOW:
    skyflow_f1 = headline_f1("skyflow", ALL_IDS)
    if skyflow_f1 is not None:
        rows.append(("skyflow alone", skyflow_f1, None))

# Composite local-only.
rows.append(("composite local-only (upper-bound)", headline_f1("ensemble_local_full", ALL_IDS), None))
rows.append(("composite local-only (honest)", headline_f1("ensemble_local_holdout", SCORE_IDS), None))

# Composite all-in (if Skyflow present).
if INCLUDE_SKYFLOW:
    rows.append(("composite all-in (upper-bound)", headline_f1("ensemble_all_full", ALL_IDS), None))
    rows.append(("composite all-in (honest)", headline_f1("ensemble_all_holdout", SCORE_IDS), None))

# Render as a Markdown table.
schema_cols = [label for _, label in SCHEMA_DISPLAY]
delta_cols = [f"Δ {label}" for _, label in SCHEMA_DISPLAY] if skyflow_f1 else []
head = ["row"] + schema_cols + delta_cols
lines = ["| " + " | ".join(head) + " |", "|" + "|".join(["---"] * len(head)) + "|"]

def _fmt(v):
    return "—" if v is None else f"{v:.3f}"

for name, f1, annot in rows:
    cells = [name]
    if f1 is None:
        cells.extend(["—"] * (len(schema_cols) + len(delta_cols)))
        lines.append("| " + " | ".join(cells) + " |")
        continue
    for schema, _ in SCHEMA_DISPLAY:
        cell = _fmt(f1[schema])
        if annot and schema in annot:
            cell = f"{cell} ({annot[schema]})"
        cells.append(cell)
    if skyflow_f1:
        for schema, _ in SCHEMA_DISPLAY:
            delta = f1[schema] - skyflow_f1[schema]
            sign = "+" if delta >= 0 else ""
            cells.append(f"{sign}{delta:.3f}")
    lines.append("| " + " | ".join(cells) + " |")

display(Markdown("### Headline F1 comparison\n\n" + "\n".join(lines)))

## 11. Per-category bar chart

Visualise which categories each composite picks up. X-axis: dataset canonical labels. One bar per row, grouped per label. Type-schema F1.

Reading the chart:
- Where the composite bars **exceed** their constituents, the recipe is doing real work.
- Where the composite bar **dips below** a constituent, it's the nervaluate cross-tag artifact discussed in the workshop notebook (Caveat 2).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

series = []

# Best individual local per category (not per-schema; per-label Type F1).
best_local_per_label = {lbl: 0.0 for lbl in _DATASET_LABELS}
best_local_who = {lbl: None for lbl in _DATASET_LABELS}
for det in LOCAL_DETECTORS:
    f1s = per_label_f1(det, ALL_IDS)
    for lbl, f in f1s.items():
        if f > best_local_per_label[lbl]:
            best_local_per_label[lbl] = f
            best_local_who[lbl] = det
series.append(("best individual local", best_local_per_label))

if INCLUDE_SKYFLOW:
    series.append(("skyflow alone", per_label_f1("skyflow", ALL_IDS)))

series.append(("composite local-only (upper)", per_label_f1("ensemble_local_full", ALL_IDS)))
series.append(("composite local-only (honest)", per_label_f1("ensemble_local_holdout", SCORE_IDS)))
if INCLUDE_SKYFLOW:
    series.append(("composite all-in (upper)", per_label_f1("ensemble_all_full", ALL_IDS)))
    series.append(("composite all-in (honest)", per_label_f1("ensemble_all_holdout", SCORE_IDS)))

x = np.arange(len(_DATASET_LABELS))
bar_w = 0.8 / max(len(series), 1)
fig, ax = plt.subplots(figsize=(max(14, 1.0 * len(_DATASET_LABELS)), 5.5))
for i, (name, by_label) in enumerate(series):
    vals = [by_label.get(lbl, 0.0) for lbl in _DATASET_LABELS]
    ax.bar(x + i * bar_w, vals, bar_w, label=name)
ax.set_xticks(x + bar_w * (len(series) - 1) / 2)
ax.set_xticklabels(_DATASET_LABELS, rotation=20)
ax.set_ylabel("Per-category F1 (Type schema)")
ax.set_title(f"Composite vs individual detectors — {DATASET}, n={len(ALL_IDS)}")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", ncol=2, fontsize=8)
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

print("\nbest-local-per-label attribution:")
for lbl, who in best_local_who.items():
    if who is None:
        continue
    print(f"  {lbl:<14} -> {who}  (F1={best_local_per_label[lbl]:.3f})")

## 12. Discussion

Things to read off this run:

1. **Did the local-only composite close the Skyflow gap?** Compare `composite local-only (honest)` row to `skyflow alone` in the headline table. If the gap is small per-schema, a local-only deployment is viable for cost-sensitive use cases.
2. **Where does Skyflow win?** Look at the recipe in section 9 — the categories where Skyflow appears are categories no local detector matches. Those are the high-value categories for cloud routing.
3. **Where do locals beat Skyflow?** Categories where a local detector won the recipe in the all-in composite. Even if you ship Skyflow as the primary, routing those specific labels to a local detector saves API calls + may bump F1.
4. **How much does the recipe drift between upper-bound and honest?** The drift count printed in sections 8 and 9 — large drift means the recipe is overfitting to particular fixtures, and you want more data before locking in a production recipe.
5. **Honest vs upper-bound F1 gap.** The honest column is what production will see; the upper-bound is the ceiling. Big gap = bigger variance, more data needed.

### Next steps for the API

Once a recipe is stable across runs, the composite can be wrapped as a regular detector for the `opf-api` `POST /api/find` endpoint. The recipe ships as configuration; clients pick `detector: composite_<name>` like any other. See the longer write-up in the main workshop notebook about conflict resolution strategies for ambiguous spans (recipe-priority, calibrated confidence, pattern validators, LLM arbiter).